# 02 — Correlations & Fraud-vs-Normal Signals

Building on notebook 01, this notebook answers three questions:

1. **How correlated are the V features?** They came from PCA — they should be near-uncorrelated by construction.
2. **Does `Amount` leak into the V space?** Amount wasn't part of the PCA. If a V feature correlates strongly with Amount, it's a redundancy worth knowing about.
3. **Which features separate fraud from normal the most?** We rank by Cohen's *d* — a per-feature effect size that's robust to the class imbalance.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fraud_shield.config import settings
from fraud_shield.data.schema import V_COLUMNS, validate

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 40)

In [ ]:
csv_path = settings.data_raw / "creditcard.csv"
assert csv_path.exists(), f"Run `make data` first — {csv_path} not found"

df = validate(pd.read_csv(csv_path))
df.shape

## Correlation structure of V1..V28

By PCA construction these should be uncorrelated with each other. Anything strongly off-diagonal would indicate the dataset was post-processed (sampled, undersampled) in a way that broke the orthogonality.

In [ ]:
v_corr = df[list(V_COLUMNS)].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    v_corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("V1..V28 pairwise correlation")
plt.tight_layout()
plt.show()

# Worst off-diagonal correlation (sanity check)
off_diag = v_corr.where(~np.eye(28, dtype=bool)).abs().stack()
off_diag.nlargest(5).round(4)

## Correlation of V features with `Amount` and `Time`

Amount wasn't part of the PCA — if it leaks into a V dimension, that feature is essentially a noisy proxy for Amount.

In [ ]:
raw_cols = ["Time", "Amount"]
cross_corr = df[list(V_COLUMNS) + raw_cols].corr().loc[list(V_COLUMNS), raw_cols]

fig, ax = plt.subplots(figsize=(4, 9))
sns.heatmap(
    cross_corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    annot=True, fmt=".2f",
    cbar_kws={"shrink": 0.7},
    ax=ax,
)
ax.set_title("V*..Amount/Time")
plt.tight_layout()
plt.show()

cross_corr.abs().max().round(3)

## Per-feature fraud-vs-normal comparison

For each feature we compute:
- mean and std for the fraud and non-fraud groups
- **Cohen's *d*** = (μ_fraud − μ_normal) / pooled_std

Cohen's *d* gives the standardized mean difference. Convention: |d| ≥ 0.2 small, ≥ 0.5 medium, ≥ 0.8 large. It's a good effect-size summary that ignores absolute scale, which matters because the V features all have variance ≈ 1 but Amount doesn't.

In [ ]:
def cohens_d(series: pd.Series, target: pd.Series) -> float:
    """Standardized mean difference between target==1 and target==0."""
    a = series[target == 1]
    b = series[target == 0]
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    if pooled == 0:
        return 0.0
    return float((a.mean() - b.mean()) / pooled)

feature_cols = list(V_COLUMNS) + ["Amount", "Time"]
stats = (
    pd.DataFrame({
        "mean_fraud": [df.loc[df["Class"] == 1, c].mean() for c in feature_cols],
        "mean_normal": [df.loc[df["Class"] == 0, c].mean() for c in feature_cols],
        "cohens_d": [cohens_d(df[c], df["Class"]) for c in feature_cols],
    }, index=feature_cols)
    .assign(abs_d=lambda x: x["cohens_d"].abs())
    .sort_values("abs_d", ascending=False)
)
stats.head(15).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ranked = stats.sort_values("abs_d", ascending=True)
colors = ["#dd8452" if d < 0 else "#4c72b0" for d in ranked["cohens_d"]]
ax.barh(ranked.index, ranked["cohens_d"], color=colors)
ax.axvline(0, color="black", linewidth=0.6)
ax.axvline(0.5, color="#888", linewidth=0.6, linestyle="--")
ax.axvline(-0.5, color="#888", linewidth=0.6, linestyle="--")
ax.set_xlabel("Cohen's d  (positive = elevated in fraud)")
ax.set_title("Per-feature effect size, ranked")
plt.tight_layout()
plt.show()

In [ ]:
top4 = stats.head(4).index.tolist()
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, top4):
    sns.boxplot(data=df, x="Class", y=col, ax=ax,
                palette=["#4c72b0", "#dd8452"], showfliers=False)
    ax.set_title(f"{col}  (|d|={stats.loc[col, 'abs_d']:.2f})")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## Observations

*(Fill in after running against the real dataset.)*

- **V-V correlations** are near-zero (PCA is doing its job).
- **V-Amount correlations** are tiny across the board — Amount is genuinely orthogonal to the PCA features, so we can keep both.
- **Top separators** are typically V14, V17, V12, V10 with |d| > 1 (large effect). Those will dominate any tree split early in training.
- **Time** is a weak signal on its own; we'll engineer `hour` and possibly `time_since_first_seen` features in the next notebook.
- **Amount** has a moderate effect but heavy tails — a `log1p` transform is justified before any linear baseline.

## Next

- **Day 4** — stratified and time-aware train/val/test splits in `fraud_shield.data.splits`.
- **Day 5** — engineered features (`hour`, `log1p(Amount)`, rolling counts) as scikit-learn transformers in `fraud_shield.features.transformers`.